**Code-only release candidate.** Market inputs and executed outputs are deliberately absent pending redistribution clearance. See `DATA_SCHEMA.md` before running market-dependent cells. `python src/teaching_interfaces.py` runs deterministic synthetic checks without market data; it does not reproduce the empirical results.

# Weekly Equity Learning and Quantum Kernel Controls

## Goal

Companion to Chapter 3. Compare classical models, a product fidelity kernel, a rotation/re-uploading family and a small variational classifier. The target is next-week membership of the top four in a fixed eight-stock cohort. Four economic features use four qubits; eight securities determine observations, not the feature-register width. Regime detection is explained in the chapter as a related extension, not reported as an executed experiment here.

## Setup

Run this notebook from top to bottom. The experiment uses local simulation only; no cloud account, trading connection or API key is needed. Market observations stop on **31 August 2026**. The supplied data are quoted-price proxies, not verified total-return series.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

# Resolve only the package itself; no author-specific absolute path is needed.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data" / "daily_market.csv").exists(), "Authorised daily inputs are required; see DATA_SCHEMA.md. Synthetic checks run without market data via src/teaching_interfaces.py."
sys.path.insert(0, str(ROOT / "src"))
from common import DATA, RESULTS, market


## 1. Execute chronological model selection

Features use the prior completed week. The year 2024 trains initial models; 2025 selects among pre-specified configurations. Final models refit the latest 52 eligible pre-2026 weeks. All 2026 results remain out of selection. Complete-window rules exclude six target weeks after the missing AAPL observation.

In [ ]:
from quantum_learning import run
summary = run()
display(pd.Series({k: summary[k] for k in ["training_2024", "validation_2025", "refit", "test", "test_weeks", "test_last_label_end"]}, name="Split sizes").to_frame())
display(pd.DataFrame(summary["selected"])[["method", "C", "scale", "layers", "auc"]])

## 2. Examine the financial evidence

AUC pools observations while IC ranks the eight names within each week. No seed is selected on the final test. The depth-two model is a separate diagnostic, not a replacement for the validation-selected one-layer map.

In [ ]:
metrics = pd.read_csv(RESULTS / "qml_metrics.csv")
display(metrics[["method", "auc", "balanced_accuracy", "mean_ic", "mean_top2"]].round(4))
display(Image(filename=str(RESULTS / "qml_comparison.png")))

## 3. Check classical equivalence

A terminal fixed entangler cancels from a fidelity kernel. Both the product kernel and the selected one-layer two-axis kernel have explicit classical formulas. This checks the representation, not forecasting performance.

In [ ]:
display(pd.Series({"Product identity error": summary["product_identity_max_error"], "Two-axis identity error": summary["two_axis_identity_max_error"]}, name="Circuit checks").to_frame())

## 4. Resample complete weeks, preserving model pairing

The 224 stock-week rows contain only 28 complete weeks. Each resampled week carries all eight stocks, labels, returns and frozen model scores. Pooled AUC is recomputed, not averaged from within-week AUCs. Five thousand paired replicates use 1-, 2- and 4-week blocks. Longer blocks must be consecutive calendar weeks: they cannot wrap or bridge the excluded period. The same indices are used for every model. Model fitting and earlier selection are not repeated; edge weighting and the short non-stationary sample limit interpretation.

In [ ]:
uncertainty = json.loads((RESULTS / "qml_uncertainty.json").read_text())
intervals = pd.read_csv(RESULTS / "qml_auc_intervals.csv")
display(intervals[intervals.block_weeks == 4].round(4))
display(pd.DataFrame(uncertainty["sensitivity"]).round(4))
display(pd.DataFrame(uncertainty["vqc_seed_summary"]).T.round(4))
print("VQC dispersion is sample SD across three specified initialisations, not a confidence interval.")

## Interpretation and Next Steps

The selected kernel's point estimates exceed the RBF baseline. The four-week AUC-difference interval excludes zero, but the independent-week interval includes zero; report both rather than choose the favourable result. Paired IC intervals include zero. None establishes stable predictive or quantum computational advantage. The one-layer fidelity has a classical closed form. VQC mean and sample SD summarise all three fixed seeds, not a selected best seed or an ensemble. A falling training loss is not out-of-sample validation.